# PhoWhisper LoRA fine-tune — Kaggle runner

Config-driven pipeline. `src/` never hardcodes a Kaggle path — this notebook is the
only place `/kaggle/input/...` appears, passed in via `--override`.

**Before running**: attach as Kaggle Dataset inputs (Add Data):
- `paid-dataset` (from `D:/phowhisper-finetune-exp/dataset/paid-dataset` locally)
- `real-meetings-bench` (from `dataset/real-meetings-bench/` in this repo, ~80 MB,
  produced by `scripts/ingest_real_bench.py` — zip and upload as a Kaggle Dataset)
- (optional) a GPU accelerator (T4 x1 is enough — see handoff, batch 8 measured at 9.75 GiB)

VIVOS (OOD) does not need a Kaggle Dataset attachment — `scripts/fetch_vivos.py`
downloads it directly from HF Hub in Cell 4.

Run cells **in order**, stopping to read output at each stage before continuing —
this pipeline has never run end-to-end; do not queue all cells blind.

## 1. Clone / update the repo

In [ ]:
import os

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Confirm the exact mount paths before setting the overrides below — Kaggle slugs the
dataset name, so this can differ from what you expect.

In [ ]:
!ls -la /kaggle/input

## 3. Set platform-specific paths

Edit these three to match what Cell above printed. This is the *only* place a
`/kaggle/input/...` path is written — everything downstream goes through
`--override`, never a hardcoded path inside `src/`.

In [ ]:
DATASET_PATH = "/kaggle/input/paid-dataset"           # edit to match Cell 2's listing
REAL_BENCH_PATH = "/kaggle/input/real-meetings-bench"  # edit to match Cell 2's listing
OOD_EVAL_PATH = "/kaggle/working/Reworkwhisper-finetune/dataset/vivos"  # written by Cell 4

OVERRIDES = (
    f"--override data.dataset_path={DATASET_PATH} "
    f"--override data.real_bench_path={REAL_BENCH_PATH} "
    f"--override data.ood_eval_path={OOD_EVAL_PATH}"
)
print(OVERRIDES)

## 4. Fetch VIVOS (OOD benchmark)

**Untested end-to-end before this run** — parquet route primary, tarball fallback.
Read the printed schema before trusting the manifest it writes.

In [ ]:
!python scripts/fetch_vivos.py --out dataset/vivos --smoke

In [ ]:
# If the smoke run above looks right, fetch the full test split (no --smoke / --limit):
!python scripts/fetch_vivos.py --out dataset/vivos

## 5. Stage: smoke

CPU-only, no model download. Proves config load, manifest merge, split resolution,
normalization, and the peft compat patch all work on this exact Kaggle image before
any GPU time is spent. **This has never run on Kaggle before — read the output
carefully, do not assume it just works.**

In [ ]:
!python -m src.pipeline --stage smoke {OVERRIDES}

## 6. Stage: baseline

Base model over test + OOD + real bench. Writes `metrics/baseline.json` and
`audit/predictions_baseline_*.csv`. **Only run this after Cell 5 (smoke) is clean.**

In [ ]:
RUN_ID = "v0-r16"  # matches configs/experiment.yaml:run_id unless overridden here
!python -m src.pipeline --stage baseline --override run_id={RUN_ID} {OVERRIDES}

In [ ]:
import json
print(json.dumps(json.load(open(f"outputs/{RUN_ID}/metrics/baseline.json")), indent=2))

## 7. Stage: train

LoRA SFT, rank from `configs/experiment.yaml` (16). **UNVERIFIED**: the
`Trainer(eval_dataset=dict)` multi-eval-set API this depends on has not been
exercised in this repo. Watch the first few log lines closely for an import or
argument error before letting it run the full 3 epochs.

In [ ]:
!python -m src.pipeline --stage train --override run_id={RUN_ID} {OVERRIDES}

## 8. HF token (only needed if you intend to push in Cell 9)

Add `HF_TOKEN` under this notebook's Add-ons → Secrets first. Never hardcode the
token here — it must not end up in any committed artifact.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print(f"No HF_TOKEN secret configured ({e}) -- fine if hub.push is false")

## 9. Stage: sweep-gate

λ sweep (hard-fails if no λ fits `sweep.ood_cer_budget` — no fallback) → gate tiers
1/2/4a → HF push iff `overall_pass` and `hub.push: true`. Set `hub.push`/`hub.repo_id`
below only once you've decided to actually publish — this is an outward-facing action.

In [ ]:
HUB_PUSH = False        # flip to True only when ready to publish
HUB_REPO_ID = None       # e.g. "your-username/phowhisper-lora-v0-r16"

hub_overrides = f"--override hub.push={HUB_PUSH} " + (f"--override hub.repo_id={HUB_REPO_ID} " if HUB_REPO_ID else "")
!python -m src.pipeline --stage sweep-gate --override run_id={RUN_ID} {OVERRIDES} {hub_overrides}

In [ ]:
import json
print(json.dumps(json.load(open(f"outputs/{RUN_ID}/metrics/gate_results.json")), indent=2))

## 10. Evidence — CER + predictions

Everything under `outputs/{run_id}/` is the run's evidence: `metrics/baseline.json`,
`metrics/lambda_sweep.csv`, `metrics/gate_results.json`, and every
`audit/predictions_*.csv` (segment-level ref/hyp for baseline and gate, per tier).
Download this whole folder before the Kaggle session ends — it is not saved anywhere
else.

In [ ]:
!find outputs/{RUN_ID} -type f | sort